In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

DATA_DIR = Path("/content/drive/MyDrive/PlantDiseaseProject/PlantVillage")
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("Exists:", DATA_DIR.exists())
subdirs = sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()]) if DATA_DIR.exists() else []
print("Subdirectories count:", len(subdirs))
print("First 10 subdirs:", subdirs[:10])

Exists: True
Subdirectories count: 1
First 10 subdirs: ['Plant_leave_diseases_dataset_without_augmentation']


In [ ]:
records = []
corrupted_files = []

for path in sorted(DATA_DIR.rglob("*")):
    if path.is_file() and path.suffix.lower() in IMAGE_EXTS:
        class_name = path.parent.name
        try:
            with Image.open(path) as img:
                img.verify()
            with Image.open(path) as img_rgb:
                w, h = img_rgb.size
                mode = img_rgb.mode
            records.append({
                "path": str(path),
                "class_name": class_name,
                "width": w,
                "height": h,
                "mode": mode,
                "file_size_kb": round(path.stat().st_size / 1024, 2),
            })
        except Exception as exc:
            corrupted_files.append((str(path), str(exc)))

df = pd.DataFrame(records)
print(f"Total valid images: {len(df)}")
print(f"Corrupted images: {len(corrupted_files)}")
print(f"Unique classes: {df['class_name'].nunique() if not df.empty else 0}")
print("\nTop 5 classes:\n", df["class_name"].value_counts().head(5))
print("\nBottom 5 classes:\n", df["class_name"].value_counts().tail(5))

In [ ]:
class_counts = df["class_name"].value_counts()
imbalance_ratio = class_counts.iloc[0] / class_counts.iloc[-1]
print(f"Max / Min class count ratio: {imbalance_ratio:.2f}x")
print("Image modes:\n", df["mode"].value_counts())
print("\nTop 5 (width, height):\n", df.groupby(["width", "height"]).size().sort_values(ascending=False).head(5))

dup_mask = df.duplicated(subset=["class_name", "file_size_kb", "width", "height"], keep=False)
print(f"Potential duplicate candidates: {int(dup_mask.sum())}")

plt.figure(figsize=(14, 5))
class_counts.plot(kind="bar")
plt.title("PlantVillage Class Distribution")
plt.xlabel("Class Name")
plt.ylabel("Number of Images")
plt.xticks(rotation=90, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
sample_classes = df["class_name"].drop_duplicates().sample(n=min(6, df["class_name"].nunique()), random_state=42)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

for ax, cls in zip(axes, sample_classes):
    sample_row = df[df["class_name"] == cls].iloc[0]
    with Image.open(sample_row["path"]) as img:
        ax.imshow(img.convert("RGB"))
    ax.set_title(cls[:28], fontsize=9)
    ax.axis("off")

for ax in axes[len(sample_classes):]:
    ax.axis("off")

plt.tight_layout()
plt.show()